# ADVC Q4 checks: trainable-parameter diff + per-epoch PGD sweep (TinyImageNet)

**Before running (right sidebar):**
1. Settings -> Accelerator -> **GPU T4 x2**
2. Settings -> Internet -> **On**
3. Add Input -> **jainmanav20/advc-results** (checkpoints) and **albert1994/tinyimagenet** (images)

Then **Run All**. Total time: about 1.5 h (dataset prep ~5 min, check 1 ~2 min, quick test ~5 min, full sweep ~1 h on two T4s).

Everything is read-only for your inputs; results go to `/kaggle/working/` (Output tab). At the end, copy the output of the **last cell** back to Claude.

In [ ]:
# 1. GPU check + clone the code
import os, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
if not os.path.isdir('/kaggle/working/ADVC'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Jmanav/ADVC.git', '/kaggle/working/ADVC'], check=True)
os.chdir('/kaggle/working/ADVC')
print('cwd:', os.getcwd())
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
cfg_text = open('configs/base.yaml').read()
assert 'name: \"tiny-imagenet\"' in cfg_text, 'configs/base.yaml dataset.name must be tiny-imagenet'
print('config dataset = tiny-imagenet: OK')

In [ ]:
# 2. Libraries
!pip install -q timm torchattacks bitsandbytes
import torch, timm, torchattacks, bitsandbytes
print('torch', torch.__version__, '| timm', timm.__version__, '| torchattacks', torchattacks.__version__, '| bitsandbytes', bitsandbytes.__version__)
print('GPUs visible:', torch.cuda.device_count())

In [ ]:
# 3. Link the TinyImageNet checkpoints from advc-results to where the code expects them
import glob, os
hits = glob.glob('/kaggle/input/**/at_int8_epoch01.pt', recursive=True)
print('found:', hits)
assert hits, 'No checkpoints found: attach jainmanav20/advc-results'
CKPT_ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs('results/checkpoints', exist_ok=True)
if not os.path.exists('results/checkpoints/tiny-imagenet'):
    os.symlink(CKPT_ROOT, 'results/checkpoints/tiny-imagenet')
for sub in ['base', 'at', 'atkd']:
    p = os.path.join('results/checkpoints/tiny-imagenet', sub)
    print(sub, '->', sorted(os.listdir(p))[:3], '...', len(os.listdir(p)), 'files')

In [ ]:
# 4. Build /kaggle/working/tiny_if (ImageFolder layout) from albert1994/tinyimagenet
import os, glob, subprocess, sys
SRC = '/kaggle/input/datasets/albert1994/tinyimagenet/tiny-imagenet-200'
if not os.path.isdir(SRC):
    cands = [p for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True)
             if os.path.isfile(os.path.join(p, 'val', 'val_annotations.txt'))]
    assert cands, 'TinyImageNet not found: attach albert1994/tinyimagenet'
    SRC = cands[0]
OUT = '/kaggle/working/tiny_if'
print('Source:', SRC)
r = subprocess.run([sys.executable, 'scripts/prepare_tiny_imagenet.py', '--root', SRC, '--out', OUT],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
print('exit code:', r.returncode)
for d in ['val_if', 'train_if']:
    p = os.path.join(OUT, d)
    print(d, '->', len(os.listdir(p)) if os.path.isdir(p) else 'MISSING', 'class folders')
assert os.path.isdir(os.path.join(OUT, 'val_if')), 'val_if was not built: see output above'

In [ ]:
# 5. Dataset check: same classes as July, standard counts, readable 64x64 images, labels match the fine-tuned model
import os, hashlib, torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from models.loader import load_config, load_model
from utils.datasets import build_eval_transform, build_remapped_folder
VAL, TRAIN = '/kaggle/working/tiny_if/val_if', '/kaggle/working/tiny_if/train_if'
JULY_FINGERPRINT = '4ccf9058e1031c01c7490930bfe2645a31f461e39adefc12096ea452b6780d50'
DATASET_OK = True
def check(cond, msg):
    global DATASET_OK
    DATASET_OK &= bool(cond)
    print(('PASS ' if cond else 'FAIL ') + msg)
classes = sorted(d for d in os.listdir(VAL) if os.path.isdir(os.path.join(VAL, d)))
check(len(classes) == 200, f'val has 200 class folders (found {len(classes)})')
check(hashlib.sha256(','.join(classes).encode()).hexdigest() == JULY_FINGERPRINT, 'class list matches July fingerprint')
val_counts = [len(os.listdir(os.path.join(VAL, c))) for c in classes]
check(sum(val_counts) == 10000 and set(val_counts) == {50}, f'val = 10,000 images, 50 per class (found {sum(val_counts)})')
n_train = sum(len(os.listdir(os.path.join(TRAIN, c))) for c in os.listdir(TRAIN))
check(n_train == 100000, f'train = 100,000 images (found {n_train})')
bad = []
for c in classes:
    f = os.path.join(VAL, c, sorted(os.listdir(os.path.join(VAL, c)))[0])
    try:
        with Image.open(f) as im:
            if im.size != (64, 64):
                bad.append((f, im.size))
    except Exception as e:
        bad.append((f, repr(e)))
check(not bad, f'sample images open and are 64x64 ({len(bad)} problems) {bad[:2]}')
cfg = load_config('configs/base.yaml')
ds = build_remapped_folder(VAL, build_eval_transform(cfg), cfg)
idx = list(range(0, len(ds), 25))
model = load_model('deit_small', 'fp32', cfg, device='cuda').eval()
correct = n = 0
with torch.no_grad():
    for x, y in DataLoader(Subset(ds, idx), batch_size=64, num_workers=2):
        correct += (model(x.cuda()).argmax(1).cpu() == y).sum().item(); n += len(y)
check(correct / n > 0.75, f'fine-tuned FP32 clean accuracy {correct / n:.3f} on {n} images (July full-set value: 0.857)')
del model; torch.cuda.empty_cache()
print('==> DATASET OK' if DATASET_OK else '==> DATASET PROBLEM: stop and send this output to Claude')
assert DATASET_OK

In [ ]:
%%writefile check_trainable.py
# Check 1 (CPU): which tensors changed between epoch 1 and epoch 7 of each TinyImageNet run.
import argparse, json, re
from pathlib import Path
import torch


def load(p):
    return torch.load(str(p), map_location='cpu', weights_only=False)


def tensors(obj):
    if isinstance(obj, dict):
        return {k: v for k, v in obj.items() if torch.is_tensor(v)}
    return {k: v for k, v in obj.state_dict().items() if torch.is_tensor(v)}


def compare(first, last):
    a, b = tensors(load(first)), tensors(load(last))
    changed, groups, numel = [], {}, 0
    for k in sorted(set(a) & set(b)):
        x, y = a[k], b[k]
        if not (x.shape == y.shape and x.dtype == y.dtype and torch.equal(x, y)):
            changed.append({'key': k, 'dtype': str(y.dtype), 'numel': y.numel()})
            numel += y.numel()
            g = re.sub('blocks[.][0-9]+[.]', 'blocks.N.', k)
            groups[g] = groups.get(g, 0) + 1
    bias = sorted({str(v.dtype) for k, v in b.items() if k.endswith('.bias')})
    return {'n_tensors': len(b), 'n_changed': len(changed), 'changed_numel': numel,
            'groups': groups, 'bias_dtypes': bias, 'changed': changed}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--ckpt-root', default='results/checkpoints/tiny-imagenet')
    ap.add_argument('--out', default='/kaggle/working/check_trainable.json')
    args = ap.parse_args()
    root = Path(args.ckpt_root)
    report = {}
    for defense in ('at', 'atkd'):
        for prec in ('fp32', 'int8', 'int4'):
            suffix = '_full_model.pt' if prec == 'int4' else '.pt'
            first = root / defense / f'{defense}_{prec}_epoch01{suffix}'
            last = root / defense / f'{defense}_{prec}_epoch07{suffix}'
            name = f'{defense}/{prec}'
            try:
                r = compare(first, last)
            except Exception as exc:
                report[name] = {'error': repr(exc)}
                print(name, 'ERROR', repr(exc))
                continue
            report[name] = r
            print(f'{name:10s} changed {r["n_changed"]:4d}/{r["n_tensors"]:4d} tensors, {r["changed_numel"]:>10,} values; bias dtypes {r["bias_dtypes"]}')
    Path(args.out).write_text(json.dumps(report, indent=2))
    print('written', args.out)


if __name__ == '__main__':
    main()

In [ ]:
# 6. Run check 1 (already done once; repeated here so this notebook is self-contained, ~2 min)
!python check_trainable.py

In [ ]:
%%writefile pgd_epoch_sweep.py
# Check 2 (GPU): clean / FGSM / PGD-20 accuracy after every training epoch of the TinyImageNet AT and AT+KD runs.
# Resumable: rows already in --out are skipped. Fixed class-stratified subset; every attack call seeded.
import argparse, json, sys, time
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
sys.path.insert(0, str(Path.cwd()))
import attacks.fgsm as fgsm_mod
import attacks.pgd as pgd_mod
from models.loader import load_config, load_model
from utils.datasets import build_eval_transform, build_remapped_folder


class Logits(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        out = self.m(x)
        return out.logits if hasattr(out, 'logits') else out


def stratified_indices(ds, per_class, seed):
    g = torch.Generator().manual_seed(seed)
    by_class = {}
    for i, (_, y) in enumerate(ds.samples):
        by_class.setdefault(y, []).append(i)
    idx = []
    for y in sorted(by_class):
        members = by_class[y]
        pick = torch.randperm(len(members), generator=g)[:per_class].tolist()
        idx.extend(members[j] for j in pick)
    return idx


_FRESH_INT4 = None


def fill_missing_attrs(loaded, fresh):
    """The int4 checkpoints are pickled whole models saved with the July timm. A newer timm's
    forward() reads attributes (e.g. Attention.gate) the old objects never had. Copy ONLY simple
    missing attributes (None / bool / int / float / str, or a missing Identity/None submodule) from a
    freshly built model of the same config; anything else is refused so no weights are ever changed."""
    fresh_mods = dict(fresh.named_modules())
    added = set()
    for name, m in loaded.named_modules():
        f = fresh_mods.get(name)
        if f is None or type(f) is not type(m):
            continue
        for attr, val in vars(f).items():
            if attr.startswith('_') or attr in vars(m):
                continue
            if val is None or isinstance(val, (bool, int, float, str)):
                setattr(m, attr, val)
                added.add(f'{type(m).__name__}.{attr}={val!r}')
            else:
                raise RuntimeError(f'refusing to fill {name}.{attr} of type {type(val).__name__}')
        for k, sub in f._modules.items():
            if k not in m._modules:
                if sub is None or isinstance(sub, nn.Identity):
                    m._modules[k] = None if sub is None else nn.Identity()
                    added.add(f'{type(m).__name__}.{k}={type(sub).__name__}')
                else:
                    raise RuntimeError(f'refusing to fill submodule {name}.{k} of type {type(sub).__name__}')
    return sorted(added)


def load_checkpoint(cfg, prec, defense, epoch, root):
    global _FRESH_INT4
    if epoch > 0 and prec == 'int4':
        path = root / defense / f'{defense}_{prec}_epoch{epoch:02d}_full_model.pt'
        try:
            m = torch.load(str(path), map_location='cuda', weights_only=False)
        except Exception as exc:
            print(f'[sweep] cuda load failed ({exc!r}); retrying via cpu', flush=True)
            m = torch.load(str(path), map_location='cpu', weights_only=False).cuda()
        if _FRESH_INT4 is None:
            _FRESH_INT4 = load_model('deit_small', 'int4', cfg, device='cuda')
        added = fill_missing_attrs(m, _FRESH_INT4)
        if added:
            print(f'[sweep] {path.name}: filled missing attrs from current timm: {added}', flush=True)
        return m.eval()
    model = load_model('deit_small', prec, cfg, device='cuda')
    if epoch > 0:
        path = root / defense / f'{defense}_{prec}_epoch{epoch:02d}.pt'
        model.load_state_dict(torch.load(str(path), map_location='cuda'))
    return model.eval()


@torch.no_grad()
def n_correct(model, x, y):
    return (model(x).argmax(1) == y).sum().item()


def evaluate(model, loader, cfg, seed):
    model = Logits(model).eval()
    fgsm = fgsm_mod.build_attack(model, cfg)
    pgd = pgd_mod.build_attack(model, cfg)
    n = clean = r_fgsm = r_pgd = 0
    for b, (x, y) in enumerate(loader):
        x, y = x.cuda(), y.cuda()
        n += y.numel()
        clean += n_correct(model, x, y)
        torch.manual_seed(seed + b)
        r_fgsm += n_correct(model, fgsm(x, y), y)
        torch.manual_seed(seed + b)
        r_pgd += n_correct(model, pgd(x, y), y)
    return {'n': n, 'clean_acc': clean / n, 'fgsm_robust_acc': r_fgsm / n, 'pgd20_robust_acc': r_pgd / n}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--ckpt-root', default='results/checkpoints/tiny-imagenet')
    ap.add_argument('--out', default='/kaggle/working/pgd_epoch_sweep.jsonl')
    ap.add_argument('--per-class', type=int, default=3)
    ap.add_argument('--precisions', default='fp32,int8,int4')
    ap.add_argument('--defenses', default='at,atkd')
    ap.add_argument('--seed', type=int, default=0)
    args = ap.parse_args()
    cfg = load_config('configs/base.yaml')
    assert cfg['dataset']['name'] == 'tiny-imagenet'
    ds = build_remapped_folder(cfg['dataset']['val_dir'], build_eval_transform(cfg), cfg)
    idx = stratified_indices(ds, args.per_class, args.seed)
    loader = DataLoader(Subset(ds, idx), batch_size=cfg['eval']['batch_size'], shuffle=False, num_workers=2)
    out = Path(args.out)
    done = set()
    if out.is_file():
        for line in out.read_text().splitlines():
            r = json.loads(line)
            done.add((r['defense'], r['precision'], r['epoch']))
    root = Path(args.ckpt_root)
    jobs = []
    for prec in args.precisions.split(','):
        jobs.append(('none', prec, 0))
        for defense in args.defenses.split(','):
            jobs.extend((defense, prec, e) for e in range(1, 8))
    for defense, prec, epoch in jobs:
        if (defense, prec, epoch) in done:
            continue
        t0 = time.time()
        model = load_checkpoint(cfg, prec, defense, epoch, root)
        res = evaluate(model, loader, cfg, args.seed)
        res.update(defense=defense, precision=prec, epoch=epoch, seconds=round(time.time() - t0, 1),
                   torch=torch.__version__, gpu=torch.cuda.get_device_name(0))
        with out.open('a') as f:
            f.write(json.dumps(res) + '\n')
        print(f'{defense:5s} {prec:5s} ep{epoch} clean={res["clean_acc"]:.3f} fgsm={res["fgsm_robust_acc"]:.3f} pgd20={res["pgd20_robust_acc"]:.3f} ({res["seconds"]}s)', flush=True)
        del model
        torch.cuda.empty_cache()


if __name__ == '__main__':
    main()

In [ ]:
# 7. Quick test: FP32 only, 200 images (~5 min). Should print 15 lines; 'none fp32 ep0' clean should be ~0.8-0.9
!python pgd_epoch_sweep.py --per-class 1 --precisions fp32 --out /kaggle/working/sweep_quick.jsonl

In [ ]:
# 8. Full sweep: 600 images, all precisions. Uses both T4s in parallel if available (~1 h). Resumable.
import subprocess, sys, os, time, torch
if torch.cuda.device_count() >= 2:
    plan = {'gpu0': ('0', 'fp32,int8'), 'gpu1': ('1', 'int4')}
else:
    plan = {'gpu0': ('0', 'fp32,int8,int4')}
procs = {}
for name, (gpu, precs) in plan.items():
    log = open(f'/kaggle/working/sweep_{name}.log', 'a')
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
    procs[name] = subprocess.Popen([sys.executable, 'pgd_epoch_sweep.py', '--precisions', precs,
                                    '--out', f'/kaggle/working/sweep_{name}.jsonl'],
                                   stdout=log, stderr=subprocess.STDOUT, env=env)
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    for name in procs:
        lines = open(f'/kaggle/working/sweep_{name}.log').read().splitlines()
        print(time.strftime('%H:%M'), name, '|', lines[-1] if lines else '(starting)', flush=True)
print('exit codes:', {n: p.returncode for n, p in procs.items()})
for name in procs:
    tail = open(f'/kaggle/working/sweep_{name}.log').read().splitlines()[-15:]
    print('--- last lines of', name); print('\n'.join(tail))

In [ ]:
# 9. SUMMARY: copy everything this cell prints back to Claude
import json, glob
rows = []
for f in sorted(glob.glob('/kaggle/working/sweep_gpu*.jsonl')):
    rows += [json.loads(l) for l in open(f) if l.strip()]
order = {'fp32': 0, 'int8': 1, 'int4': 2}
rows.sort(key=lambda r: (order[r['precision']], r['defense'] != 'none', r['defense'], r['epoch']))
print('precision defense epoch  n   clean   fgsm   pgd20')
for r in rows:
    print(f"{r['precision']:9s} {r['defense']:7s} {r['epoch']:5d} {r['n']:4d}  {r['clean_acc']:.3f}  {r['fgsm_robust_acc']:.3f}  {r['pgd20_robust_acc']:.3f}")
print(len(rows), 'rows (expected 45) | env:', rows[0]['torch'] if rows else '-', rows[0]['gpu'] if rows else '-')
ct = json.load(open('/kaggle/working/check_trainable.json'))
for k, v in ct.items():
    print('check1', k, v.get('error') or (v['n_changed'], v['changed_numel'], v['bias_dtypes']))